# Phase 4: Hybrid NCA-Transformer (Kaggle GPU Runner)

This notebook trains and evaluates the Phase 4 architectures:
1. **Hybrid NCA-Transformer (~10.58M params, +3.47% overhead)**
   - Pre-attention weight-shared cellular adaptor stem ($K=2$ microsteps, receptive field $RF=7$, full GRU channel gating).
   - 3-layer Transformer decoder backbone ($d=384, h=6$, RoPE, causal self-attention).
2. **Hybrid CNN-Transformer Control (~10.58M params, matched control)**
   - Pre-attention unshared 2-layer causal convolutional filter matched in parameter count and receptive field ($RF=7$).
3. **Rigorous Gate 4 Evaluation Protocol**
   - **Clean Perplexity Benchmark:** Tests if the hybrid preserves the Transformer's ~42 PPL generative power.
   - **Probe 4A: Surface Noise Robustness Sweep:** Measures degradation slope $\beta$ across in-vocabulary noise $p \in [0.0..0.20]$.
   - **Probe 4B: Impulse Perturbation Attenuation:** Measures cumulative damage area $D$ under internal shock.
   - **Gate 4 Verdict Protocol:** Formally verifies whether the cellular hybrid delivers local noise resilience without hurting clean PPL.

> **Prerequisite:** Ensure Accelerator is set to **GPU (T4 x1 or P100)** in the right sidebar.

In [ ]:
# Cell 1: Environment Setup, Clone/Update Repository & Prepare Data
import os, sys
if not os.path.exists('/kaggle/working/NCA-sim'):
    !git clone https://github.com/Zenoguy/NCA-sim.git /kaggle/working/NCA-sim

%cd /kaggle/working/NCA-sim
!git fetch origin main
!git checkout main
!git reset --hard origin/main

!pip install -q tokenizers pyyaml matplotlib pytest

# Ensure WikiText-2 data is downloaded and tokenized
if not os.path.exists('data/raw/train.npy'):
    from data.prepare_wikitext2 import prepare_wikitext2
    from data.tokenizer import BPETokenizerWrapper
    splits = prepare_wikitext2()
    tok = BPETokenizerWrapper('data/tokenizer.json')
    for s in ['train', 'valid', 'test']:
        tok.encode_file(splits[s])

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Verify Groundwork & Run Unit Test Suite
!pytest tests/ -v

## Train Phase 4 Hybrid Models on GPU (10 Epochs Each)

In [ ]:
# Cell 3: Train Primary Hybrid NCA-Transformer (Stem K=2, d_adaptor=160)
!python train.py --config configs/level4_hybrid_nca.yaml

In [ ]:
# Cell 4: Train Matched-Control Hybrid CNN-Transformer (2-Layer Conv, d_adaptor=160)
!python train.py --config configs/level4_hybrid_cnn_control.yaml

## Run Phase 4 Probes & Gate 4 Protocol on GPU

In [ ]:
# Cell 5: Execute Full Probing Suite & Gate 4 Evaluation
!python scripts/run_level4.py --action all --device cuda

## Generate Publication Figures

In [ ]:
# Cell 6: Generate Publication Figures (Robustness Curves, Clean PPL, Perturbation)
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

out_dir = Path("outputs/level4")
eval_file = out_dir / "hybrid_evaluation.json"

if eval_file.exists():
    with open(eval_file) as f:
        data = json.load(f)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

    # Subplot 1: Clean Test Perplexity Comparison
    ax1 = axes[0]
    clean_data = data.get("clean_ppl", {})
    models = list(clean_data.keys())
    ppls = [clean_data[m].get("test_ppl", 0.0) for m in models]
    labels = ["Pure Transformer", "Hybrid NCA (K=2)", "Hybrid CNN Control"]
    colors = ["#d62728", "#1f77b4", "#2ca02c"]

    bars = ax1.bar(range(len(models)), ppls, color=colors, width=0.55, edgecolor="black", alpha=0.85)
    ax1.set_xticks(range(len(models)))
    ax1.set_xticklabels(labels, rotation=15, ha="right", fontsize=10, fontweight="bold")
    ax1.set_ylabel("Test Perplexity (WikiText-2)", fontsize=11)
    ax1.set_title("Clean Language Modeling Performance", fontsize=12, fontweight="bold")
    for bar in bars:
        yval = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2.0, yval + 0.8, f"{yval:.2f}", ha="center", va="bottom", fontweight="bold")

    # Subplot 2: Surface Input Noise Relative Degradation R(p)
    ax2 = axes[1]
    rob_data = data.get("robustness", {})
    for m, col, lab in zip(models, colors, labels):
        m_rob = rob_data.get(m, {})
        curve = m_rob.get("curve", [])
        beta = m_rob.get("degradation_slope_beta", 0.0)
        if curve:
            p_vals = [pt["corruption_rate_p"] for pt in curve]
            r_vals = [pt["relative_degradation_ratio"] for pt in curve]
            ax2.plot(p_vals, r_vals, marker="o", lw=2.2, label=f"{lab} (β={beta:.2f})", color=col)
    ax2.set_xlabel("In-Vocab Noise Corruption Rate (p)", fontsize=11)
    ax2.set_ylabel("Relative Degradation Ratio R(p)", fontsize=11)
    ax2.set_title("Probe 4A: Surface Noise Robustness", fontsize=12, fontweight="bold")
    ax2.legend(fontsize=10)

    # Subplot 3: Impulse Perturbation Attenuation Trajectory
    ax3 = axes[2]
    pert_data = data.get("perturbation", {})
    for m, col, lab in zip(models, colors, labels):
        m_pert = pert_data.get(m, {})
        traj = m_pert.get("trajectory_subsequent_delta", [])[:28]
        D = m_pert.get("cumulative_damage_area", 0.0)
        if traj:
            ax3.plot(range(65, 65 + len(traj)), traj, lw=2.0, label=f"{lab} (D={D:.2f})", color=col)
    ax3.set_xlabel("Sequence Token Position (t)", fontsize=11)
    ax3.set_ylabel("Error Delta ΔL_t", fontsize=11)
    ax3.set_title("Probe 4B: Impulse Perturbation Attenuation", fontsize=12, fontweight="bold")
    ax3.legend(fontsize=10)

    plt.tight_layout()
    fig_path = out_dir / "phase4_hybrid_evaluation.png"
    plt.savefig(fig_path, dpi=300)
    plt.show()
    print(f"Publication figure saved to: {fig_path}")
else:
    print("Evaluation artifact not found. Please run Cell 5 first.")

## Archive Phase 4 Artifacts for Instant Download

In [ ]:
# Cell 7: Package JSON summaries and figures (tiny, < 1MB, zero .pt model files)
!tar -czvf /kaggle/working/outputs_level4.tar.gz outputs/level4/*.json outputs/level4/*.png
print("\nPhase 4 outputs successfully archived to /kaggle/working/outputs_level4.tar.gz (instant download ready!)")